In [ ]:
# Custom
import sys
sys.path.insert(0, '../src')
from args import args
import utils


## Utils and Inference
import csv
from collections import Counter
import json
import numpy as np
import pandas as pd
import re

import os
from tqdm import tqdm

In [ ]:
import sys
sys.path.insert(0, '..')
from prompts import app_name_only_prompts

In [ ]:
class args(args):

    experimental_setting = "app_name_only"
    
    setting = ["batching", "single_variable"][1]
    saving_path = f"../saved_data/inference/{setting}/{experimental_setting}"
    
    variable_column = "variable_name"
    app_name_column = "app_name"
    
    data_path  = '../data/get_variable_para_info_formatted_info_annotation_set_23_06_2026_with_path.json'

    platform = ["llm_provider_1", "watsonx", "mse", "litellm", "azure"][0]

In [ ]:
## Check if the saving directories exist or not; If not, then create them
utils.create_save_directories()

In [ ]:
### Check if the experiments saving directories exist or not; If not, then create them
utils.ensure_exp_dirs_exist(args.saving_path)

In [ ]:
prompt = app_name_only_prompts.app_name_only_prompt_v1
print(prompt)

### Data Preprocessing

In [ ]:
data_df = pd.read_json(args.data_path)

In [ ]:
variable_names = list(data_df[args.variable_column])

### Validating and Loading the Inference Pipeline

In [ ]:
inference = utils.Inference(args)

In [ ]:
model_names = utils.validate_api_endpoints(args, args.platform)

print(f"Models from {args.platform}: {model_names}\n" )


model_names = ['llama3_3_70b', 'gpt_oss_120b', 'gpt_oss_20b',  'qwen_3_8B', 'llama4_17b_16e']

print(f"Selected Models from {args.platform}: {model_names}" )

In [ ]:
pred_folder=args.saving_path+"/"
pred_folder

In [ ]:

class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super().default(obj)

def write_json_record(file_path: str, record: dict) -> None:
    """Append a record to a JSON array file."""
    with open(file_path, 'r+') as jsonfile:
        data = json.load(jsonfile)
        data.append(record)
        jsonfile.seek(0)
        json.dump(data, jsonfile, indent=2, cls=NumpyEncoder)
        jsonfile.truncate()

In [ ]:
file_names = []

for model_name in tqdm(model_names, desc="Models", position=0):
    
    prediction_df = data_df
    file_name = f"{model_name}_variable_DD_{args.experimental_setting}_.json"
    
    file_name = utils.update_file_name(file_name, pred_folder=pred_folder, extension=".json")

    
    # Checkpointing Logic- model_name can be a string representing model identifier. 
    # The model_name and file_name are intended to be manually defined to resume checkpointing. This assume that the file is already created.
    
    if model_name == '':
        file_name = ''
        checkpoint_index = 10
        
    else:
        checkpoint_index = 0
        with open(pred_folder + file_name, 'w') as jsonfile:
            json.dump([], jsonfile)

    
    print(f"Starting Inference for {model_name} in {file_name}")
    pbar = tqdm(total=len(variable_names), desc=f"Inference [{model_name}]",
                position=1, leave=False, dynamic_ncols=True)
    

    full_path = pred_folder + file_name
    
    fields = list(prediction_df.columns) + ["prompt_used", "token_statistics", "predictions_DD"]

    for i, var_name in enumerate(variable_names):
        if i < checkpoint_index:
            pbar.update(1)
            continue

        if i == checkpoint_index:
            print(f"starting inference from index {i}")

        row_data = list(prediction_df.iloc[i])
        current_prompt = prompt[:]

        # Inject the application name as the only context
        app_name = prediction_df.iloc[i][args.app_name_column]
        current_prompt = current_prompt.replace("{{APP_NAME}}", f"{app_name}")
        current_prompt = current_prompt.replace("{{TEST_VARIABLE}}", f"{var_name}")

        response = None
        token_stats = None

        try:
            response, raw_output = inference(prompt=current_prompt, inference_platform=args.platform,
                                             model_name=model_name, return_raw=True)
            token_stats = dict(dict(raw_output)['usage'])

            # Filtering For serialization
            token_stats = {k: token_stats[k] for k in ('completion_tokens', 'prompt_tokens', 'total_tokens')}
            
        except Exception as e:
            if not response and not token_stats:
                response = token_stats = f"error found: {e}"

            else: 
                response = str(response)
                token_stats = str(response)

        record = dict(zip(fields, row_data + [current_prompt, token_stats, response]))
        write_json_record(full_path, record)
        pbar.update(1)

    pbar.close()
    file_names.append(file_name)